In [1]:
import os
import sys
import glob
from pathlib import Path
from astropy.io import fits
import numpy as np
from numpy import savetxt
import matplotlib
import matplotlib.pyplot as plt
from PyAstronomy import pyasl
from scipy.interpolate import interp1d
from astropy import units as u
from astropy.coordinates import SkyCoord
#install dustmaps from https://github.com/gregreen/dustmaps
from dustmaps.sfd import SFDQuery
import pandas as pd
from astropy.table import Table
import gc
matplotlib.use("Agg")
plt.ioff()
from PIL import Image
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.cbook import get_sample_data
from astroquery.sdss import SDSS
import urllib.request
import warnings
from joblib import Parallel, delayed

In [2]:
storage_dir = os.path.join(os.path.expanduser("~"),'Data_storageHII/')
os.makedirs(storage_dir, exist_ok=True)

In [3]:
tex_folder = storage_dir + '/specTx_DR7/'
plots_folder =  storage_dir + '/specPlt_DR7/'


if (os.path.exists(tex_folder) == True and os.path.exists(plots_folder) == True):
    print('Folder for TEX set saving already created \n')
    print('Folder for PLOTS set saving already created \n')
else:
    if (os.path.exists(plots_folder) == False):
        print(f'Folder for PLOTS set saving created ({plots_folder}) \n')
        os.makedirs(plots_folder, exist_ok=True)
    if (os.path.exists(tex_folder) == False):
        print(f'Folder for TEX set saving created ({tex_folder}) \n')
        os.makedirs(tex_folder, exist_ok=True)
    else:
        print(f'Folder for TEX set saving created ({tex_folder}) \n')
        print(f'Folder for PLOTS set saving created ({plots_folder}) \n')
        os.makedirs(tex_folder, exist_ok=True)
        os.makedirs(plots_folder, exist_ok=True)




Folder for TEX set saving already created 

Folder for PLOTS set saving already created 



In [4]:
fits_folder = storage_dir + '/HIIG_specDR7/'
spec_list = os.listdir(fits_folder)
spec_list.remove('DOWNspec_listDR7.txt')
spec_list = sorted(spec_list)
len(spec_list)

121

In [5]:
def sdss_SpectralSynthesis_prepare_DR7(NAME,fits_folder,tex_folder):
    hdu = fits.open(fits_folder+NAME)
    data = hdu[0].data
    ##############################################################
    #Read the header
    
    zz = hdu[0].header['Z']
    RA = hdu[0].header['RA']
    DEC = hdu[0].header['DEC']
    
    c = SkyCoord(ra=RA*u.degree, dec=DEC*u.degree, frame='icrs')
    c.galactic
    
    sfd = SFDQuery()
    ebv01 = sfd(c.galactic)
    
    ##############################################################
    #Read the spectra, flux and wavelength
    
    flux = data[0]
    wav00 = hdu[0].header['COEFF0']
    wavdiff = hdu[0].header['COEFF1']

    hdu.close()
    
    waveair = 10**(wav00 + wavdiff*np.arange(len(flux)))
    
    waveobs = waveair/(1.0+2.735182E-4+131.4182/(waveair)**2 +2.76249E8/(waveair)**4)
    
    ##############################################################
    #redshift correction
    
    wave = waveobs/(1.0+zz)
    
    ##############################################################
    #Unredding the spectra
    
    fluxUnred = pyasl.unred(wave, flux, ebv=ebv01, R_V=3.1)
    
    ##############################################################
    #cosmoiogical flux corrextion
    
    flux_final = fluxUnred*(1.0+zz)**3
    
    ##############################################################
    #Change the wavelength to starlight input
    
    wave_in = int(wave[2])             #initial wavelength
    wave_fin = int(wave[len(wave)-2])  #final wavelength
    
    wave_new = np.arange(wave_in, wave_fin, 1)
    
    flux_interpolate = interp1d(wave, flux_final, kind='quadratic')
    flux_starlight = flux_interpolate(wave_new)
    
    ##############################################################
    #Crate an output array
    
    table = list(zip(wave_new, flux_starlight))
    
    # Save the file

    new_name  = tex_folder   + (NAME.replace('.fit',"")) 
    savetxt(new_name + '.tex', table, fmt='%8.4f', delimiter='    ')
    
    return f"{NAME} spectra saved in {new_name} .tex"

In [6]:
tmp = [sdss_SpectralSynthesis_prepare_DR7(t,fits_folder,tex_folder) for t in spec_list]

In [7]:
HIIG_IMG_folder = storage_dir + '/HIIG_img/'


if (os.path.exists(HIIG_IMG_folder) == True ):
    print('Folder for HIIG SDSS images set saving already created \n')
else:
    print(f'Folder for HIIG SDSS images set saving created ({HIIG_IMG_folder}) \n')
    os.makedirs(HIIG_IMG_folder, exist_ok=True)

Folder for HIIG SDSS images set saving already created 



In [8]:
catalog_file = os.path.join(storage_dir,"gal_info_dr7_v5_2.fit.gz")
catalog = catalog_file
catalog=Table.read(catalog)
w=((catalog['RA']!=catalog['DEC']) & (catalog['RA']>0) & ((catalog['DEC']>=-90) | (catalog['DEC']<=90)) & (catalog['Z']>=0) & ( catalog['Z'] != float('nan') ) & ( catalog['DEC'] != float('nan') ) & ( catalog['RA'] != float('nan') ))
reduced_cat=catalog[w]

In [10]:
def HIIG_image_retriever(NAME,fits_folder,img_folder,catalog_table):
    warnings.filterwarnings('ignore')
    reduced_cat=catalog_table

    hdu = fits.open(fits_folder+NAME)
    
    PLATE = hdu[0].header['PLATEID']
    FIBER = hdu[0].header['FIBERID']
    MJD = hdu[0].header['MJD']

    IDs = SDSS.query_specobj(plate = int(PLATE), mjd = int(MJD), fiberID = int(FIBER))

    status = str(type(IDs))

    if (status != "<class 'NoneType'>"):
        ra = float(SDSS.query_specobj(plate = int(PLATE), mjd = int(MJD), fiberID = int(FIBER))['ra'])
        dec = float(SDSS.query_specobj(plate = int(PLATE), mjd = int(MJD), fiberID = int(FIBER))['dec'])
    else:
        ra = np.array(reduced_cat[(reduced_cat['PLATEID']==int(PLATE)) & (reduced_cat['FIBERID']==int(FIBER)) & (reduced_cat['MJD']==int(MJD))]['RA'])[0]
        dec = np.array(reduced_cat[(reduced_cat['PLATEID']==int(PLATE)) & (reduced_cat['FIBERID']==int(FIBER)) & (reduced_cat['MJD']==int(MJD))]['DEC'])[0]



    imgURL = f"http://skyserver.sdss.org/dr16/SkyServerWS/ImgCutout/getjpeg?TaskName=Skyserver.Explore.Image&ra={ra}&dec={dec}&scale=0.2&width=200&height=200&opt=G"

    result = SDSS.query_specobj(plate=PLATE,mjd=MJD,fiberID=FIBER,fields = ['specobjid'])

    status = str(type(result))

    if (status != "<class 'NoneType'>"):
        result = str(result['specobjid'][0])
        image_spec = f'https://skyserver.sdss.org/dr16/en/get/SpecById.ashx?id={result}'
        if os.path.exists(f"{img_folder}/specimage_{NAME}.jpeg")!=True:
            urllib.request.urlretrieve(image_spec,f"{img_folder}/specimage_{NAME}.jpeg")
        alert = 0
    else:
        alert = 'No hay imagen de este espectro'



    hdu.close()

    

    if os.path.exists(f"{img_folder}/{NAME}.jpeg")!=True:
        urllib.request.urlretrieve(imgURL,f"{img_folder}/{NAME}.jpeg")
    




    return f"{PLATE},{FIBER},{MJD},{ra},{dec},{alert}\n"


#tmp = Parallel(n_jobs=-1)(delayed(HIIG_image_retriever)(e,fits_folder,HIIG_IMG_folder,reduced_cat) for e in spec_list)

for e in range(len(spec_list)):
    tmp = e + 0 #e + 94 
    print(HIIG_image_retriever(spec_list[tmp],fits_folder,HIIG_IMG_folder,reduced_cat),e)

267,421,51608,147.597162628828,0.708127177268213,0
 0
341,606,51690,200.94776134765,-1.54777983724568,0
 1
388,457,51793,1.73758544441037,0.857230431344045,0
 2
415,141,51810,53.8609151908011,-0.636533804183749,0
 3
381,370,51811,346.765651605092,1.21977224962501,0
 4
382,328,51816,348.675639921919,1.10584233786806,0
 5
410,220,51816,43.6088418917947,-0.689572212220896,0
 6
418,302,51817,8.07750453626596,15.0039393153258,0
 7
400,441,51820,23.4359526850297,0.953121139115853,0
 8
429,495,51820,26.779343875627,13.9414254674982,0
 9
384,281,51821,352.402311298105,-1.18250049873833,0
 10
447,361,51877,131.365038079905,53.1480248071366,0
 11
442,156,51882,126.377825178692,50.8012690367536,0
 12
481,483,51908,148.112334702567,2.29996410813119,0
 13
270,617,51909,153.62905412036,0.798627952656554,0
 14
485,550,51909,142.52681944382,60.4481509142727,0
 15
275,445,51910,161.478244311513,1.06827527952586,0
 16
456,195,51910,41.2236197598615,-8.36054102570387,0
 17
456,306,51910,40.217485051174,-

In [11]:
hdu = fits.open(fits_folder+spec_list[0])
hdu[0].header

SIMPLE  =                    T                                                  
BITPIX  =                  -32                                                  
NAXIS   =                    2                                                  
NAXIS1  =                 3818                                                  
NAXIS2  =                    4                                                  
TAI     =        4458951910.50 / 1st row - Number of seconds since Nov 17 1858  
RA      =            147.73167 / 1st row - Right ascension of telescope boresigh
DEC     =           -0.033455  / 1st row - Declination of telescope boresight (d
EQUINOX =              2000.00 /                                                
RADECSYS= 'FK5     '           /                                                
AZ      =        113.863787882 / 1st row - Azimuth  (encoder) of tele (0=N?) (de
ALT     =        56.6194209010 / 1st row - Altitude (encoder) of tele        (de
AIRMASS =        1.201784234

In [12]:
def spec_plotter(name,
                figsize=(48, 18),        # << mucho más pequeño
                render_dpi=100,          # DPI de canvas
                save_dpi=100,            # DPI del archivo final
                tol=1.0,                  # tolerancia en Å para localizar líneas           
                img_folder='',
                fits_folder='',
                tex_folder='',
                plots_folder=''):

    ll = ['HB','O[III]a','O[III]b','Ha']
    lines = {
    "HB": [4862.721, r'H-$\beta$', 4862.0000],
    "O[III]a": [4960.295, r'O[III]$\lambda$4959', 4960.0000],
    "O[III]b": [5008.239, r'O[III]$\lambda$5007', 5007.0000],
    "Ha": [6562.801, r'H-$\alpha$', 6562.0000]
    }



    # Construye paths/nombres como en tu versión
    #base = (name.replace('.fit', '')) + ".tex"
    #base = base.replace(".tex", f"._{str(counter)}_.tex")

    base_fit = fits_folder + name
    base_tex = tex_folder + name.replace('.fit', '.tex')
    base_img = img_folder + name.replace('.fit', '.fit.jpeg')



    hdu = fits.open(base_fit)
    
    plateid,fiberid,mjd,Z,Zerr,ra,dec = hdu[0].header['PLATEID'],hdu[0].header['FIBERID'],hdu[0].header['MJD'],hdu[0].header['Z'],hdu[0].header['Z_ERR'],hdu[0].header['RAOBJ'],hdu[0].header['DECOBJ']

    hdu.close()

    data = Table.read(base_tex, format='ascii', fast_reader=True)
    lam = np.asarray(data['col1'], dtype=float)
    flux = np.asarray(data['col2'], dtype=float)

    fig, ax = plt.subplots(figsize=figsize, dpi=render_dpi)
    ax.spines.right.set_visible(False)
    ax.spines.top.set_visible(False)

    # Trazado principal
    ax.plot(lam, flux, label='HIIG Flux', lw=2.0, color = 'green')

    ax.set_ylabel(r"$F_{\lambda}\ \left[10^{-17}\ {\rm erg\ s}^{-1}\ {\rm cm}^{-2}\ \AA^{-1}\right]$",
                    style='oblique', family='serif', size = 35)
    ax.set_xlabel(r'Rest Wavelength [$\AA$]',
                    style='oblique', family='serif', size = 35)
    
    # Anotación de líneas (con tolerancia en lugar de igualdad exacta)
    for key in ll:
        rest = lines[key][2]
        idx = np.where(np.abs(lam - rest) <= 3)[0]
        if idx.size:
            i0 = idx[len(idx)//2]
            i1, i2 = max(i0 - 10, 0), min(i0 + 10, flux.size)
            selec_flux_max = float(flux[i1:i2].max()) + 0.05 * (flux.max() - flux.min())
            x = rest - (100 if key in ('O[III]a', 'O[III]b') else 30)
            ax.text(x, selec_flux_max, f"{lines[key][1]}",
                    fontsize=30,
                    bbox={'facecolor': '#F4F1BB', 'alpha': 0.5, 'boxstyle': "round,pad=0.2", 'ec': 'none'})
    
    ax.tick_params(axis='x', labelsize=20)
    ax.tick_params(axis='y', labelsize=20)

    ax.tick_params(axis='x', labelrotation=90)
    ax.set_xlim(3500, 7500)
    #ax.set_ylim(0, max(float(flux.max()), 1.0) * 1.05)
    ax.set_yscale('log')
    ax.grid(True, which="both", ls=":", linewidth=0.9)

    sym = [r"$\alpha$",r"$\delta$",r"$z$",r"$\mathcal{PLATE}$",r"$\mathcal{MJD}$",r"$\mathcal{FIBERID}$",r"$\mathbf{name}$",r"$\pm$"]

    leg_title = f'{sym[6]}: {name}\n {sym[0]}: {ra}°\n {sym[1]}: {dec}°\n {sym[2]}: {Z}{sym[7]}{Zerr}\n {sym[3]}: {plateid}\n {sym[4]}: {mjd}\n {sym[5]}: {fiberid}\n'

    #leg_title = f"PLATEID = {plateid}\nMJD = {mjd}\nFIBERID = {fiberid}\nZ = {Z}\n"
    ax.legend(title=leg_title, loc='lower left', prop={'size': 20}, title_fontsize=30)


    image_path = plt.imread(get_sample_data(base_img))


    inset_ax = inset_axes(ax, width="30%", height="30%",loc='upper left')
    inset_ax.imshow(image_path)

    inset_ax.axis('off')



    os.makedirs(plots_folder, exist_ok=True)
    outfile = os.path.join(plots_folder, name.replace('.fit', '.png'))

    # Guarda (si "tight" no es imprescindible, quítalo para ahorrar un poco más)
    fig.savefig(outfile, dpi=save_dpi, bbox_inches='tight', transparent=False)

    # Cierra y limpia SOLO esta figura
    plt.close(fig)
    del fig, ax, data, lam, flux
    gc.collect()
    return f"Done for {name.replace('.fit', '.png')} "

spec_plotter(spec_list[50],img_folder=HIIG_IMG_folder,fits_folder=fits_folder,tex_folder=tex_folder,plots_folder=plots_folder)    


'Done for spSpec-52261-0741-279.png '

In [13]:
tmp = Parallel(n_jobs=-1)(delayed(spec_plotter)(e,img_folder=HIIG_IMG_folder,fits_folder=fits_folder,tex_folder=tex_folder,plots_folder=plots_folder) for e in spec_list)

In [14]:
hdu = fits.open(fits_folder+spec_list[93])
hdu[0].header

SIMPLE  =                    T                                                  
BITPIX  =                  -32                                                  
NAXIS   =                    2                                                  
NAXIS1  =                 3829                                                  
NAXIS2  =                    4                                                  
TAI     =        4609853174.00 / 1st row - Number of seconds since Nov 17 1858  
RA      =            161.78680 / 1st row - Right ascension of telescope boresigh
DEC     =            13.844803 / 1st row - Declination of telescope boresight (d
EQUINOX =              2000.00 /                                                
RADECSYS= 'FK5     '           /                                                
AZ      =        26.5012390656 / 1st row - Azimuth  (encoder) of tele (0=N?) (de
ALT     =        68.3792181922 / 1st row - Altitude (encoder) of tele        (de
AIRMASS =        1.085474049

In [22]:
PLATE = hdu[0].header['PLATEID']
FIBER = hdu[0].header['FIBERID']
MJD = hdu[0].header['MJD']
print(PLATE,FIBER,MJD)
#IDs = SDSS.query_specobj(plate = int(PLATE), mjd = int(MJD), fiberID = int(FIBER))

1749 26 53357


In [23]:
imgURL = f"http://skyserver.sdss.org/dr16/SkyServerWS/ImgCutout/getjpeg?TaskName=Skyserver.Explore.Image&ra=162.787042&dec=13.324442&scale=0.2&width=200&height=200&opt=G"
imgURL

'http://skyserver.sdss.org/dr16/SkyServerWS/ImgCutout/getjpeg?TaskName=Skyserver.Explore.Image&ra=162.787042&dec=13.324442&scale=0.2&width=200&height=200&opt=G'

In [24]:
image_spec = f'https://skyserver.sdss.org/dr16/en/get/SpecById.ashx?id=1969206140214470656'


urllib.request.urlretrieve(image_spec,f"{HIIG_IMG_folder}/specimage_{spec_list[93]}.jpeg")

('/home/holman/Data_storageHII//HIIG_img//specimage_spSpec-53357-1749-026.fit.jpeg',
 <http.client.HTTPMessage at 0x70dd56a16fd0>)

In [ ]:
for e in range(len(download_list)): #len(download_list)
    A = download_list[e][64:-4]
    MJD = A.partition('-')[0]
    PLATE = (A.partition('-')[2]).partition('-')[0]
    FIBER = (A.partition('-')[2]).partition('-')[2]

    IDs = SDSS.query_specobj(plate = int(PLATE), mjd = int(MJD), fiberID = int(FIBER))

    status = str(type(IDs))

    if (status != "<class 'NoneType'>"):
        ra = float(SDSS.query_specobj(plate = int(PLATE), mjd = int(MJD), fiberID = int(FIBER))['ra'])
        dec = float(SDSS.query_specobj(plate = int(PLATE), mjd = int(MJD), fiberID = int(FIBER))['dec'])
    else:
        ra = np.array(reduced_cat[(reduced_cat['PLATEID']==int(PLATE)) & (reduced_cat['FIBERID']==int(FIBER)) & (reduced_cat['MJD']==int(MJD))]['RA'])[0]
        dec = np.array(reduced_cat[(reduced_cat['PLATEID']==int(PLATE)) & (reduced_cat['FIBERID']==int(FIBER)) & (reduced_cat['MJD']==int(MJD))]['DEC'])[0]

    print(ra, dec)


    imgURL = f"http://skyserver.sdss.org/dr16/SkyServerWS/ImgCutout/getjpeg?TaskName=Skyserver.Explore.Image&ra={ra}&dec={dec}&scale=0.2&width=200&height=200&opt=G"

    #urllib.request.urlretrieve(imgURL,HOME +  '/HIIGs/SDSS_Imaging/' + f'imObj-{A}.jpeg')

    #print(imgURL)

    
    #print(ra)

In [3]:
tex_folder = '/HIIGs/Tex_spectra/'
plots_folder =  '/HIIGs/Spectra_plots/'

if (os.path.exists(HOME + tex_folder) == True and os.path.exists(HOME + plots_folder) == True):
    print('Folder for TEX set saving already created \n')
    print('Folder for PLOTS set saving already created \n')
else:
    if (os.path.exists(HOME + plots_folder) == False):
        print(f'Folder for PLOTS set saving created ({HOME+plots_folder}) \n')
        os.mkdir(HOME+plots_folder)
    if (os.path.exists(HOME + tex_folder) == False):
        print(f'Folder for TEX set saving created ({HOME+tex_folder}) \n')
        os.mkdir(HOME+plots_folder)
    else:
        print(f'Folder for TEX set saving created ({HOME+tex_folder}) \n')
        print(f'Folder for PLOTS set saving created ({HOME+plots_folder}) \n')
        os.mkdir(HOME+tex_folder)
        os.mkdir(HOME+plots_folder)

Folder for TEX set saving already created 

Folder for PLOTS set saving already created 



In [4]:
fits_folder = HOME + '/HIIGs/FITS_download/'
spec_list = os.listdir(fits_folder)
spec_list = sorted(spec_list)

In [5]:
def sdss_dust_correction(NAME,counter):
    counter = int(counter)
    hdu = fits.open(fits_folder+NAME)
    data = hdu[0].data
    ##############################################################
    #Read the header
    
    zz = hdu[0].header['Z']
    RA = hdu[0].header['RA']
    DEC = hdu[0].header['DEC']
    
    c = SkyCoord(ra=RA*u.degree, dec=DEC*u.degree, frame='icrs')
    c.galactic
    
    sfd = SFDQuery()
    ebv01 = sfd(c.galactic)
    
    ##############################################################
    #Read the spectra, flux and wavelength
    
    flux = data[0]
    wav00 = hdu[0].header['COEFF0']
    wavdiff = hdu[0].header['COEFF1']

    hdu.close()
    
    waveair = 10**(wav00 + wavdiff*np.arange(len(flux)))
    
    waveobs = waveair/(1.0+2.735182E-4+131.4182/(waveair)**2 +2.76249E8/(waveair)**4)
    
    ##############################################################
    #redshift correction
    
    wave = waveobs/(1.0+zz)
    
    ##############################################################
    #Unredding the spectra
    
    fluxUnred = pyasl.unred(wave, flux, ebv=ebv01, R_V=3.1)
    
    ##############################################################
    #cosmoiogical flux corrextion
    
    flux_final = fluxUnred*(1.0+zz)**3
    
    ##############################################################
    #Change the wavelength to starlight input
    
    wave_in = int(wave[2])             #initial wavelength
    wave_fin = int(wave[len(wave)-2])  #final wavelength
    
    wave_new = np.arange(wave_in, wave_fin, 1)
    
    flux_interpolate = interp1d(wave, flux_final, kind='quadratic')
    flux_starlight = flux_interpolate(wave_new)
    
    ##############################################################
    #Crate an output array
    
    table = list(zip(wave_new, flux_starlight))
    
    # Save the file

    C = str(counter)

    if len(C) < 3:
        if len(C) == 2:
            C = '0' + C
        if len(C) == 1:
            C = '00' + C

    new_name  = HOME + tex_folder   + '[' + C + ']' + (NAME.replace('.fit',"")) 
    savetxt(new_name + '.tex', table, fmt='%8.4f', delimiter='    ')
    print(NAME+ ' spectra saved in '+new_name + '.tex')
    

In [6]:
for i in range(len(spec_list)):
    sdss_dust_correction(sorted(spec_list)[i],i+1)

spSpec-51608-0267-421.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/[001]spSpec-51608-0267-421.tex
spSpec-51690-0341-606.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/[002]spSpec-51690-0341-606.tex
spSpec-51793-0388-457.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/[003]spSpec-51793-0388-457.tex
spSpec-51810-0415-141.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/[004]spSpec-51810-0415-141.tex
spSpec-51811-0381-370.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/[005]spSpec-51811-0381-370.tex
spSpec-51816-0382-328.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/[006]spSpec-51816-0382-328.tex
spSpec-51816-0410-220.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/[007]spSpec-51816-0410-220.tex
spSpec-51817-0418-302.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/[008]spSpec-51817-0418-302.tex
spSpec-51820-0400-441.fi

In [ ]:
# Opcional: evita trazados gigantes en Agg con paths muy largos
plt.rcParams['agg.path.chunksize'] = 10000

DATA_URL = "http://das.sdss.org/spectro/ss_tar_26/"
catalog_name = "gal_info_dr7_v5_2.fit.gz"
local_dir = HOME

# Lee el catálogo una sola vez, con mapeo de memoria
#with fits.open(os.path.join(local_dir, catalog_name), memmap=True) as hdul:
    #catalog = Table(hdul[1].data)



In [43]:
plt.rcParams['agg.path.chunksize'] = 10000

ll = ['HB','O[III]a','O[III]b','Ha']
lines = {
  "HB": [4862.721, r'H-$\beta$', 4862.0000],
  "O[III]a": [4960.295, r'O[III]$\lambda$4959', 4960.0000],
  "O[III]b": [5008.239, r'O[III]$\lambda$5007', 5007.0000],
  "Ha": [6562.801, r'H-$\alpha$', 6562.0000]
}


TEX_folder = HOME + '/HIIGs/Tex_spectra/'
tex_list = os.listdir(TEX_folder)
tex_list = sorted(tex_list)

IMG_folder = HOME + '/HIIGs/SDSS_Imaging/'
img_list = os.listdir(IMG_folder)
img_list = sorted(img_list)

Final_list = pd.read_csv(HOME + '/SPS/Final_speclist.csv', sep=",", header=0)



for u in range(1): #len(tex_list)

    u = 83

    spc = os.path.join(TEX_folder, tex_list[u])

    A = tex_list[u][12:-4]
    MJD = int(A.partition('-')[0])
    PLATE = int((A.partition('-')[2]).partition('-')[0])
    FIBER = int((A.partition('-')[2]).partition('-')[2])


    partial_list = Final_list[(Final_list['MJD'] == MJD) & (Final_list['PLATE'] == PLATE) & (Final_list['FIBERID'] == FIBER)]


    data = Table.read(spc, format='ascii', fast_reader=True)
    lam = np.asarray(data['col1'], dtype=float)
    flux = np.asarray(data['col2'], dtype=float)
    
    fig, ax = plt.subplots(figsize=(37, 12.57), dpi=50)
    ax.spines.right.set_visible(False)
    ax.spines.top.set_visible(False)
    ax.plot(lam, flux, label='HIIG Flux', lw=2.0, color = 'green')

    ax.set_ylabel(r"$F_{\lambda}\ \left[10^{-17}\ {\rm erg\ s}^{-1}\ {\rm cm}^{-2}\ \AA^{-1}\right]$",
                    style='oblique', family='serif', size = 35)
    ax.set_xlabel(r'Rest Wavelength [$\AA$]',
                    style='oblique', family='serif', size = 35)


    for key in ll:
        rest = lines[key][2]
        idx = np.where(np.abs(lam - rest) <= 3)[0]
        if idx.size:
            i0 = idx[len(idx)//2]
            i1, i2 = max(i0 - 10, 0), min(i0 + 10, flux.size)
            selec_flux_max = float(flux[i1:i2].max()) + 0.05 * (flux.max() - flux.min())
            x = rest - (100 if key in ('O[III]a', 'O[III]b') else 30)
            ax.text(x, selec_flux_max, f"{lines[key][1]}",
                    fontsize=30,
                    bbox={'facecolor': '#F4F1BB', 'alpha': 0.5, 'boxstyle': "round,pad=0.2", 'ec': 'none'})

    ax.tick_params(axis='x', labelsize=20)
    ax.tick_params(axis='y', labelsize=20)

    ax.tick_params(axis='x', labelrotation=90)
    ax.set_xlim(3500, 7000)
    ax.set_ylim(0, max(float(flux.max()), 1.0) * 1.2)
    ax.grid(True, which="both", ls=":", linewidth=0.9)


    image_path = plt.imread(get_sample_data(IMG_folder + img_list[u]))


    inset_ax = inset_axes(ax, width="60%", height="60%", loc='upper right')
    inset_ax.imshow(image_path)
    inset_ax.axis('off')

    sym = [r"$\alpha$",r"$\delta$",r"$z$",r"$\mathcal{PLATE}$",r"$\mathcal{MJD}$",r"$\mathcal{FIBERID}$",r"$\mathbf{NAME}$"]

    leg_title = f'{sym[6]}: {partial_list["Name"].iloc[0]}\n {sym[0]}: {partial_list["RA"].iloc[0]}°\n {sym[1]}: {partial_list["DEC"].iloc[0]}°\n {sym[2]}: {partial_list["Z"].iloc[0]}\n {sym[3]}: {partial_list["PLATE"].iloc[0]}\n {sym[4]}: {partial_list["MJD"].iloc[0]}\n {sym[5]}: {partial_list["FIBERID"].iloc[0]}\n'

    C = str(u+1)

    if len(C) < 3:
        if len(C) == 2:
            C = '0' + C
        if len(C) == 1:
            C = '00' + C


    new_name = '[' + f'{C}' +']' + 'SpecObj-' + A + ".png"
    ax.legend(title=leg_title, loc='upper left', prop={'size': 25}, title_fontsize=35)
    #outdir = HOME + plots_folder
    outdir = '/home/hollman/'
    os.makedirs(outdir, exist_ok=True)
    outfile = os.path.join(outdir, new_name)

    fig.savefig(outfile, dpi=50, bbox_inches='tight', transparent=False)






In [56]:
Final_list = pd.read_csv(HOME + '/SPS/Final_speclist.csv', sep=",", header=0)
Final_list

,N,Name,RA,DEC,Z,PLATE,MJD,FIBERID
0,0,UM 199,1.737614,0.857181,0.073682,388,51793,457
1,1,UM 282,12.947112,0.161124,0.037556,394,51913,472
2,2,UM 336,23.436174,0.953078,0.019236,400,51820,441
3,3,MRK 0627,131.643397,36.438978,0.010634,934,52672,369
4,4,SBS 0934+546,144.556301,54.473618,0.102102,556,51991,224
...,...,...,...,...,...,...,...,...
116,116,WISEA J103226.97+271755.3,158.112320,27.298679,0.192490,2353,53794,585
117,117,WISEA J101136.07+263027.5,152.900317,26.507660,0.054666,2347,53757,501
118,118,WISEA J162152.58+151855.9,245.469098,15.315537,0.034341,2208,53880,306
119,119,WISEA J090506.84+223834.7,136.278600,22.642759,0.125548,2284,53708,545


In [31]:
def spec_plotter(name,
                 figsize=(48, 18),        # << mucho más pequeño
                 render_dpi=100,          # DPI de canvas
                 save_dpi=100,            # DPI del archivo final
                 tol=1.0,counter=0):                # tolerancia en Å para localizar líneas

    # Construye paths/nombres como en tu versión
    base = (name.replace('.fit', '')) + ".tex"
    base = base.replace(".tex", f"._{str(counter)}_.tex")
    local_dirspc = HOME + tex_folder
    spc = os.path.join(local_dirspc, base)

    # Extrae IDs desde el nombre, como en tu código
    plateid, mjd, fiberid = [int(base[13:17]), int(base[7:12]), int(base[18:21])]

    # Selección de fila en el catálogo
    z_finder = (catalog['PLATEID'] == plateid) & (catalog['MJD'] == mjd) & (catalog['FIBERID'] == fiberid)
    reduced_cat = catalog[z_finder]
    Z = float(reduced_cat['Z'][0])

    # Lee espectro (rápido y sobrio en memoria)
    data = Table.read(spc, format='ascii', fast_reader=True)
    lam = np.asarray(data['col1'], dtype=float)
    flux = np.asarray(data['col2'], dtype=float)

    # Prepara figura/axes (OO) y estilos básicos
    fig, ax = plt.subplots(figsize=figsize, dpi=render_dpi)
    ax.spines.right.set_visible(False)
    ax.spines.top.set_visible(False)

    # Trazado principal
    ax.plot(lam, flux, label='Flux', lw=1.2)

    ax.set_ylabel(r"$F_{\lambda}\ \left[10^{-17}\ {\rm erg\ s}^{-1}\ {\rm cm}^{-2}\ \AA^{-1}\right]$",
                  style='oblique', family='serif')
    ax.set_xlabel(r'Wavelength at rest $\lambda$ [$\AA$]',
                  style='oblique', family='serif')

    # Anotación de líneas (con tolerancia en lugar de igualdad exacta)
    for key in ll:
        rest = lines[key][2]
        idx = np.where(np.abs(lam - rest) <= tol)[0]
        if idx.size:
            i0 = idx[len(idx)//2]
            i1, i2 = max(i0 - 10, 0), min(i0 + 10, flux.size)
            selec_flux_max = float(flux[i1:i2].max()) + 0.05 * (flux.max() - flux.min())
            x = rest - (100 if key in ('O[III]a', 'O[III]b') else 30)
            ax.text(x, selec_flux_max, f"{lines[key][1]}",
                    fontsize=11,
                    bbox={'facecolor': '#F4F1BB', 'alpha': 0.5, 'boxstyle': "round,pad=0.2", 'ec': 'none'})

    ax.tick_params(axis='x', labelrotation=90)
    ax.set_xlim(3500, 7500)
    ax.set_ylim(0, max(float(flux.max()), 1.0) * 1.05)
    ax.grid(True, which="both", ls=":", linewidth=0.8)

    leg_title = f"{base[:-4]}\nZ = {Z}\nPLATEID = {plateid}\nMJD = {mjd}\nFIBERID = {fiberid}"
    ax.legend(title=leg_title, loc='upper right', prop={'size': 10}, title_fontsize=11)

    outdir = HOME + plots_folder
    os.makedirs(outdir, exist_ok=True)
    outfile = os.path.join(outdir, base.replace('.tex', '') + ".png")

    # Guarda (si "tight" no es imprescindible, quítalo para ahorrar un poco más)
    fig.savefig(outfile, dpi=save_dpi, bbox_inches='tight', transparent=False)

    # Cierra y limpia SOLO esta figura
    plt.close(fig)
    del fig, ax, data, lam, flux, reduced_cat
    gc.collect()
    print('Done for ' + base.replace('.tex', ''))

In [ ]:

Final_list

,N,Name,RA,DEC,Z,PLATE,MJD,FIBERID
0,0,UM 199,1.737614,0.857181,0.073682,388,51793,457


In [ ]:
TEX_folder = HOME + '/HIIGs/Tex_spectra/'
tex_list = os.listdir(TEX_folder)
tex_list = sorted(tex_list)



# Bucle principal (usa directamente la lista)
for h in range(2): #len(spec_list)
    try:
        spec_plotter(name = sorted(tex_list)[h],counter = h+1)
    except Exception as e:
        print(f"[WARN] Falló {name}: {e}")

# Version vieja no optimizada

In [6]:
DATA_URL="http://das.sdss.org/spectro/ss_tar_26/"
catalog_name="gal_info_dr7_v5_2.fit.gz"
local_dir="/home/hollman/DataHII/"
local_file = fits.open(os.path.join(local_dir,catalog_name))
catalog=Table.read(local_file[1])
local_file.close()

ll = ['HB','O[III]a','O[III]b','Ha']
# Lines dictionary
lines = {
  "HB": [4862.721,r'H-$\beta$',4862.0000], #4862
  "O[III]a": [4960.295,r'O[III]$\lambda$4959',4960.0000],#4960
  "O[III]b": [5008.239,r'O[III]$\lambda$5007',5007.0000],#5007
  "Ha": [6562.801,r'H-$\alpha$',6562.0000] ##6550
}


In [8]:
def spec_plotter(name):
    name = (name.replace('.fit',"")) + ".tex"
    local_dirspc="/home/hollman/DataHII/HIIGalaxy_Spectra(Chavez2012)/Tex_spectra/"
    spc = os.path.join(local_dirspc,name)
    plateid, mjd, fiberid = [int(name[13:17]), int(name[7:12]), int(name[18:21])]
    z_finder = (catalog['PLATEID']==plateid) & (catalog['MJD']==mjd) & (catalog['FIBERID']==fiberid)
    reduced_cat = catalog[z_finder]
    Z = round(float(reduced_cat['Z'][0]),6)
    data = Table.read(spc, format='ascii')
    data.rename_column('col1', r'$\lambda$')
    data.rename_column('col2', 'Flux')
    plt.rcParams['axes.spines.left'] = True
    plt.rcParams['axes.spines.right'] = False
    plt.rcParams['axes.spines.top'] = False
    plt.rcParams['axes.spines.bottom'] = True
    plt.figure(figsize=(40,18),dpi=200)
    plt.plot(data[r'$\lambda$'],data['Flux'],label = 'Flux',lw=1.3, color ='#1C448E')
    plt.ylabel(r"$F_{\lambda}\ \left[ 10^{-17}\ {\rm erg\ s}^{-1}\ {\rm cm}^{-2}\ \AA^{-1} \right]$", style = 'oblique', family = 'serif', size = 25)
    plt.xlabel(r'Wavelength at rest $\lambda$ [$\AA$]', style = 'oblique', family = 'serif', size = 25)
    # Lineas a mostrar
    for a in range(len(ll)):#len(ll)
        v_lambda = lines[ll[a]][2] 
        flux_line = data[(data[r'$\lambda$'])==v_lambda]
        selec_flux_idx = np.where(data[r'$\lambda$'] == v_lambda)
        selec_flux_max = max(data['Flux'][(selec_flux_idx[0][0])-10:(selec_flux_idx[0][0])+10]) + 70
        if (a==1) or (a==2):
            v_lambda = v_lambda- 100
        else:
            v_lambda = v_lambda - 30
        plt.text(v_lambda, selec_flux_max, f'{lines[ll[a]][1]}', 
                fontsize = 25,bbox = {'facecolor': '#F4F1BB', 'alpha': 0.5, 'boxstyle': "round,pad=0.3", 'ec': 'none'}, rotation = 0)
    plt.yticks(fontsize = 20)
    plt.xticks(fontsize = 20,rotation=90)
    plt.ylim(0,max(data['Flux'],)+15)
    plt.xlim(4000,7000)
    plt.grid(True, which="both", ls=":", color = 'gray', linewidth = 0.8)
    txt_prop = {'style' : 'oblique', 'family' : 'serif', 'size' :30}       
    plt.legend(title = f'{name[:-4]} \n Z = {Z} \n PLATEID = {plateid} \n MJD = {mjd} \n FIBERID = {fiberid}', prop = txt_prop, loc= 'upper right', title_fontsize=35)
    plt.savefig("/home/hollman/DataHII/HIIGalaxy_Spectra(Chavez2012)/Spectra_plots/" + (name.replace('.tex',"")) + ".png" ,
                bbox_inches='tight', transparent=False,dpi=150)
    #plt.show()
    plt.close('all')
    print('Done for '+ (name.replace('.tex',"")))
    gc.collect()
    del name, plateid, mjd, fiberid,z_finder,reduced_cat, Z, data
    
#spSpec-52339-0578-060.tex
#spec_plotter(spec_list[0])

In [ ]:
for i in range(len(spec_list)):
    spec_plotter(spec_list[i])

Done for spSpec-52339-0578-060
Done for spSpec-52937-1269-177
Done for spSpec-53297-1781-055
Done for spSpec-52962-1585-261
Done for spSpec-53473-1695-627
Done for spSpec-52353-0507-521
Done for spSpec-53534-2112-557
Done for spSpec-51929-0490-128
Done for spSpec-54561-2708-193
Done for spSpec-51821-0384-281
Done for spSpec-53768-2372-508
Done for spSpec-54628-2318-286
Done for spSpec-51882-0442-156
Done for spSpec-53386-1872-526
Done for spSpec-52174-0637-523
Done for spSpec-52224-0564-216
Done for spSpec-52636-0999-517
Done for spSpec-51957-0502-007
Done for spSpec-53084-1758-338
Done for spSpec-52413-0976-600
Done for spSpec-52178-0640-267
Done for spSpec-53463-1981-438
Done for spSpec-52641-1003-327
Done for spSpec-53768-2373-560
Done for spSpec-52939-1582-335
Done for spSpec-51816-0382-328
Done for spSpec-53317-1921-281
Done for spSpec-53142-1697-415
Done for spSpec-52138-0652-090
Done for spSpec-52238-0566-497
Done for spSpec-53786-2356-172
Done for spSpec-52174-0664-355
Done for